In [ ]:
# Read the folders under dataset4geodiff\raw_geodiff_txt. Each folder name is the apkname, and each folder contains version_mapping.json. Read the JSON contents; the first value is version 1 and the second value is version 2.
# Then load dataset4geodiff\androzoo_gfd_app_metadata.tsv, which contains the version number for each apkname in different regions.

# Finally, build the version information for this APK across different countries.

## Read the folders under `dataset4geodiff\out_openai_sematic_geodiff_txt`; each folder name is the apkname.

Then load `dataset4geodiff\androzoo_gfd_app_metadata.tsv`, which contains the version number for each apkname in different regions.

Finally, obtain the version numbers for this APK in different countries and save them as a CSV file under the corresponding APK folder. Each apkname folder has one output file.

Iterate through each APK folder under `raw_geodiff_txt`.

Use the folder name as the apkname to match rows in `androzoo_gfd_app_metadata.tsv`.

Write one `country_versions.csv` file to each APK folder.

In [5]:
# {f_position}\{s_position}
f_position = "AA1_first_batch"
s_position = "AA4_forth_100_batch" # AA1_first_328_batch, AA4_forth_100_batch, AA6_sisth_74_batch  ## AA1_first_328_batch, AA4_forth_100_batch, AA6_sisth_74_batch

# {parameter}
# parameter first_1,  first_2,   first_third, first_4, first_5, first_6
# second_1, second_2, second_3, second_4, second_5, second_6, second_7
# third_1,  third_2,  third_3,  third_4,  third_5,  third_6,  third_7
# fourth_1
parameter = "first_6"

In [6]:
from pathlib import Path
import pandas as pd
import json

# first_1,  first_2,   first_third, first_4, first_5, first_6
# second_1, second_2, second_3, second_4, second_5, second_6, second_7
# third_1,  third_2,  third_3,  third_4,  third_5,  third_6,  third_7
# fourth_1

raw_dir = Path(f"dataset4geodiff/out_openai_sematic_geodiff_txt/{parameter}")
meta_path = Path("dataset4geodiff/androzoo_gfd_app_metadata.tsv")

In [ ]:
# Read metadata
meta_df = pd.read_csv(meta_path, sep="\t", dtype=str).fillna("")
meta_df.columns = [c.strip() for c in meta_df.columns]
meta_df["package_name"] = meta_df["package_name"].astype(str).str.strip()

In [ ]:
# Keep only country-version-related columns
country_cols = [
    c for c in meta_df.columns
    # if c == "package_name" or c.endswith("_version_name") or c.endswith("_version_code")
    if c == "package_name" or c.endswith("_version_code")
]

apk_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])

matched = 0
unmatched = 0

for apk_dir in apk_dirs:
    apkname = apk_dir.name.strip()

    # skip folders that are not APK package names
    if "." not in apkname:
        print(f"[SKIP] Not an APK folder: {apkname}")
        continue

    print(f"Processing APK: {apkname}")
    # Match the metadata row for this APK.
    row = meta_df.loc[meta_df["package_name"] == apkname, country_cols].copy()

    if row.empty:
        unmatched += 1
        # Also write an unmatched row for easier troubleshooting.
        row = pd.DataFrame([{"package_name": apkname}])
        for c in country_cols:
            if c != "package_name":
                row[c] = ""
    else:
        matched += len(row)

    # Optionally include the first two values from version_mapping.json.
    # vm_path = apk_dir / "version_mapping.json"
    # v1, v2 = "", ""
    # if vm_path.exists():
    #     try:
    #         vm_obj = json.loads(vm_path.read_text(encoding="utf-8"))
    #         vals = list(vm_obj.values())
    #         if len(vals) > 0:
    #             v1 = str(vals[0])
    #         if len(vals) > 1:
    #             v2 = str(vals[1])
    #     except Exception:
    #         pass

    # row["version_1_from_mapping"] = v1
    # row["version_2_from_mapping"] = v2

    # Write one output file per APK folder.
    out_csv = apk_dir / f"{apkname}_country_versions.csv"
    row.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"Written: {out_csv}")

print(f"Total APK folders: {len(apk_dirs)}")
print(f"Metadata rows matched: {matched}")
print(f"Unmatched APKs: {unmatched}")

## Convert `dataset4geodiff\EPP-LLM_ours_dataset_mapping.xlsx` to `clauses.json` (do not run again)
[
    
  {"ours_id": "P1", "bucket": "P", "title": "Categories of Personal Data Collected"},

  {"ours_id": "P2", "bucket": "P", "title": "Categories of PI Shared"},

In [ ]:
# import pandas as pd
# import json
# from pathlib import Path
# import re

# clauses = [
#     {"ours_id": "P1", "bucket": "P", "title": "Categories of Personal Data Collected"},
#     {"ours_id": "P2", "bucket": "P", "title": "Categories of PI Shared"},
#     # Append other clauses here...
# ]

# xlsx_path = Path("dataset4geodiff/EPP-LLM_ours_dataset_mapping.xlsx")
# out_path = Path("dataset4geodiff/clauses.json")  # Change this to the target path if needed.

# # Read Excel (the first sheet by default).
# df = pd.read_excel(xlsx_path)
# df.columns = [str(c).strip() for c in df.columns]

# # Normalize column names to support different naming conventions.
# lower_to_raw = {c.lower(): c for c in df.columns}

# def pick_col(candidates):
#     for c in candidates:
#         if c.lower() in lower_to_raw:
#             return lower_to_raw[c.lower()]
#     return None

# col_id = pick_col(["ours_id"])
# # col_bucket = pick_col(["bucket", "group", "category"])
# col_title = pick_col(["clause"])

# if col_id is None or col_title is None:
#     raise ValueError(f"Required column not found. Current columns: {df.columns.tolist()}")

# # Clean the data.
# df = df[[col_id, col_title]].copy()
# df[col_id] = df[col_id].astype(str).str.strip()
# df[col_title] = df[col_title].astype(str).str.strip()

# # Extract the bucket directly from the ours_id prefix.
# # P1 -> P, P2 -> P, CR6 -> CR, E24 -> E
# df["bucket"] = df[col_id].str.extract(r"^([A-Za-z]+)", expand=False).fillna("").str.upper()

# # Remove empty values and duplicates.
# df = df[(df[col_id] != "") & (df[col_title] != "") & (df["bucket"] != "")]
# df = df.drop_duplicates(subset=[col_id], keep="first")

# clauses = [
#     {"ours_id": r[col_id], "bucket": r["bucket"], "title": r[col_title]}
#     for _, r in df.iterrows()
# ]

# out_path.parent.mkdir(parents=True, exist_ok=True)
# out_path.write_text(json.dumps(clauses, ensure_ascii=False, indent=2), encoding="utf-8")

# print("Saved:", out_path)
# print("Total records:", len(clauses))
# print("First 2 examples:")
# print(json.dumps(clauses[:2], ensure_ascii=False, indent=2))